# 1. Adresování a správa paměti - Garbage collecting, Reference/ukazatele, Struktura paměti programu

### Struktura paměti programu
* Když operační systém spustí program, vyhradí mu v RAM virtuální adresní prostor, který se logicky dělí na několik částí:
    * Kódový segment - Zde jsou uloženy přeložené strojové instrukce programu, je obvykle určena pouze pro čtení
    * Datový segment - Pro uložení globálních a statických proměnných, které existují po celou dobu běhu programu
    * Zásobník - Paměť s architekturou LIFO, ukládají se sem lokální proměnné uvnitř funkcí a návratové adresy při zanořování do dalších funkcí, velmi rychlý, jeho velikost omezena, jakmile funkce skončí, její proměnné se ze zásobníku automaticky vymažou
    * Halda - Prostor určený pro dynamickou alokaci paměti za běhu programu, v objektových jazycích (jako je Python nebo Java) se zde vytvářejí všechny instance tříd, pole a objekty, paměť na haldě se nepromazává automaticky koncem bloku, musí se aktivně spravovat

### Ukazatele vs. Reference
* Ukazatel (C/C++) - Proměnná, jejíž hodnotou je přímá fyzická adresa jiného místa v paměti, programátor s ním může dělat matematiku a musí dávat pozor, aby neukázal do paměti cizího programu
* Reference (Python, C#, Java) - Jedná se o bezpečnější abstrakci nad ukazatelem, reference ukazuje na objekt ležící na haldě, ale programátor nemá přístup k její skutečné paměťové adrese a nemůže dělat paměťovou aritmetiku, v Pythonu neexistují klasické proměnné jako „krabice na hodnoty“, existují pouze reference, které se lepí na objekty v haldě

### Správa paměti a Garbage Collecting
* V C nebo C++ musí vývojář o paměť na haldě ručně žádat (`malloc` / `new`) a ručně vracet (`free` / `delete`), pokud zapomene, vzniká Memory Leak
* Moderní jazyky používají automatickou správu, v Pythonu funguje primárně na Reference Counting, každý objekt si pamatuje, kolik referencí na něj ukazuje, jakmile toto číslo klesne na 0, objekt se smaže
* Garbage Collector - Počítání referencí má slabinu – Cyklické reference, pokud se tyto objekty odpojí od zbytku programu, jejich počítadlo neklesne pod 1, proto na pozadí běží ještě Garbage Collector, občas zastaví program, prohledá paměť, najde tyto oddělené kusy a z paměti je uvolní

In [ ]:
import sys
import gc

**1. POČÍTÁNÍ REFERENCÍ**

In [ ]:
# Vytvoření pole na Haldě (Heap). Proměnná 'pole_a' na Stacku na něj drží referenci.
pole_a = [10, 20, 30]

# getrefcount vrací počet referencí (vždy o 1 více, protože i samotné volání
# funkce getrefcount si vytvoří dočasnou referenci ve svých argumentech)
print(f"Počet referencí na pole: {sys.getrefcount(pole_a) - 1}")

# Vytvoření další reference na stejný objekt v paměti
pole_b = pole_a
print(f"Počet referencí po přidání 'pole_b': {sys.getrefcount(pole_a) - 1}")

# Smazání referencí
del pole_a
del pole_b
# Zde počítadlo kleslo na 0 a paměť byla okamžitě uvolněna.

**2. CYKLICKÉ REFERENCE A GARBAGE COLLECTOR**

In [ ]:
class Uzel:
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self.ukazatel_soused = None

# Vytvoříme dva nezávislé uzly na haldě
uzel1 = Uzel("A")
uzel2 = Uzel("B")

# Propojíme je navzájem (Vytvoříme cyklus A -> B a B -> A)
uzel1.ukazatel_soused = uzel2
uzel2.ukazatel_soused = uzel1

# Smažeme jediné přístupové body z našeho hlavního programu
del uzel1
del uzel2

# POZOR: Objekty "A" a "B" v paměti stále fyzicky existují!
# Jejich reference counting je na hodnotě 1, protože ukazují navzájem na sebe.
# Běžný mechanismus je nesmaže. Zde nastupuje Garbage Collector.

# Ručně vynutíme běh GC (jinak by se spustil sám po určité době)
pocet_uvolnenych_objektu = gc.collect()
print(f"\nGarbage Collector prohledal haldu a uvolnil {pocet_uvolnenych_objektu} cyklicky zacyklených objektů.")